# Úkol na 14. 4. 2026

- popište krok po kroku co tento skript dělá

In [1]:
import os
import re
import nltk
from collections import defaultdict
from rapidfuzz.distance import Levenshtein

In [2]:
ROOT = os.getcwd()

BIBLE_PATH = os.path.join(ROOT, 'bible_files')

EXAMPLE_PATH = os.path.join(ROOT, 'example_corpus.txt')

In [3]:
all_verses = {}

for b_file in os.listdir(BIBLE_PATH):
    with open(os.path.join(BIBLE_PATH, b_file), 'r', encoding='utf-8') as f:
        b_data = f.read()
        
        book_dict = eval(b_data)
        all_verses.update(book_dict)

        print(len(book_dict), "verses read.... updated to ", len(all_verses), "verses in total.")

105 verses read.... updated to  105 verses in total.
437 verses read.... updated to  542 verses in total.
817 verses read.... updated to  1359 verses in total.
105 verses read.... updated to  1464 verses in total.
942 verses read.... updated to  2406 verses in total.
811 verses read.... updated to  3217 verses in total.
89 verses read.... updated to  3306 verses in total.
113 verses read.... updated to  3419 verses in total.
13 verses read.... updated to  3432 verses in total.
256 verses read.... updated to  3688 verses in total.
719 verses read.... updated to  4407 verses in total.
61 verses read.... updated to  4468 verses in total.
822 verses read.... updated to  5290 verses in total.
695 verses read.... updated to  5985 verses in total.
47 verses read.... updated to  6032 verses in total.
83 verses read.... updated to  6115 verses in total.
15 verses read.... updated to  6130 verses in total.
21 verses read.... updated to  6151 verses in total.
56 verses read.... updated to  6207 v

In [ ]:
clean_BKR_verses = {}
for verse_id, verse_text in all_verses.items():
    clean_text = "".join([c if c.isalpha() or c.isspace() else " " for c in verse_text.lower()])
    clean_BKR_verses[verse_id] = " ".join(clean_text.split())

def find_citation_in_sentence(sentence:str, window:int=5):
    verse_id_hit = False 
    
    words = nltk.word_tokenize(sentence.lower())
    clean_words = [w for w in words if w.isalpha()]

    if len(clean_words) >= window:
        for i in range(len(clean_words) - window + 1):
            phrase = " ".join(clean_words[i:i+window])
            
            for verse_id, clean_verse in clean_BKR_verses.items():
                if phrase in clean_verse:
                    verse_id_hit = verse_id
                    return verse_id_hit
                    
    return verse_id_hit


def find_citation_in_sentence_levenstein(sentence:str, window:int=5, threshold:int=3):
    verse_id_hit = False 
    
    words = nltk.word_tokenize(sentence.lower())
    clean_words = [w for w in words if w.isalpha()]

    if len(clean_words) >= window:
        for i in range(len(clean_words) - window + 1):
            phrase = " ".join(clean_words[i:i+window])
            
            for verse_id, clean_verse in clean_BKR_verses.items():
                for i in range(len(clean_verse) - window + 1):
                    verse_phrase = " ".join(clean_verse.split()[i:i+window])
                    distance = Levenshtein.distance(phrase, verse_phrase)
                    if distance <= threshold:
                        verse_id_hit = verse_id
                        print(f"Levenshtein hit: '{phrase}' ~ '{verse_phrase}' (distance: {distance})")
                        return verse_id_hit
                    
    return verse_id_hit


words_to_verses = defaultdict(list)

stop_words = ["a", "se", "je", "v", "na", "z", "o", "k", "i", "nebo", "s", "u", "nad", "pod",
             "jak", "jsou", "tedy", "tedy", "jej", "mu", "po", 'od', 'do', 'pro', 'bez', 'ale', 'ten', 'každý', 'každou',
             "ke", "bez", "ale", "ten", "každý", "každou", 'co', 'což', 'kdo', 'který', 'která', 'které', 'kterým', 'kterými', 'kterého', 'kterému', 'kterou']

for verse_id, verse_text in all_verses.items():
    words = re.findall(r'\b\w+\b', verse_text.lower())
    for word in words:
        if word not in stop_words:
            words_to_verses[word].append(verse_id)

def preselect_verses_by_words(phrase:str, threshold:int=3):
    candidate_verses = defaultdict(int)
    for word in phrase.split():
        if word in words_to_verses:
            for verse_id in words_to_verses[word]:
                candidate_verses[verse_id] += 1

    verses_with_score_greater_than_threshold = {vid: sc for vid, sc in candidate_verses.items() if sc >= threshold}
    return verses_with_score_greater_than_threshold


def find_citation_in_sentence_levenstein_preselect(sentence:str, window:int=5, threshold:int=3, candidate_threshold:int=3):
    verse_id_hit = False 
    
    words = nltk.word_tokenize(sentence.lower())
    clean_words = [w for w in words if w.isalpha()]

    if len(clean_words) >= window:
        for i in range(len(clean_words) - window + 1):
            phrase = " ".join(clean_words[i:i+window])
            
            candidate_verses = preselect_verses_by_words(phrase, threshold=candidate_threshold)
            candidate_verses = [(verse_id, clean_BKR_verses[verse_id]) for verse_id in candidate_verses]
            
            for verse_id, clean_verse in candidate_verses:
                for i in range(len(clean_verse) - window + 1):
                    verse_phrase = " ".join(clean_verse.split()[i:i+window])
                    distance = Levenshtein.distance(phrase, verse_phrase)
                    if distance <= threshold:
                        verse_id_hit = verse_id
                        print(f"Levenshtein hit: '{phrase}' ~ '{verse_phrase}' (distance: {distance})")
                        return verse_id_hit
                    
    return verse_id_hit


def load_corpus(path=EXAMPLE_PATH):
    with open(path, 'r', encoding='utf-8') as f:
        data = f.read()

    return data

def find_citations_in_corpus(corpus_path:str, window:int=5, threshold:int=2, method:str="exact"):
    corpus_text = load_corpus(corpus_path)
    sentences = nltk.sent_tokenize(corpus_text)

    hits = defaultdict(list)

    for sentence in sentences:
        if method == "exact":
            verse_id_hit = find_citation_in_sentence(sentence, window=window)
        elif method == "levenstein":
            verse_id_hit = find_citation_in_sentence_levenstein(sentence, window=window, threshold=threshold)
        elif method == "levenstein_preselect":
            verse_id_hit = find_citation_in_sentence_levenstein_preselect(sentence, window=window, threshold=threshold)
        else:
            raise ValueError("Invalid method. Choose 'exact', 'levenstein', or 'levenstein_preselect'.")

        if verse_id_hit:
            hits[verse_id_hit].append(sentence)

    return hits

In [15]:
hit = find_citation_in_sentence('Na počátku bylo slovo a to slovo bylo', window=5)
print("Exact match hit:", hit)

Exact match hit: J 1:1


In [ ]:
find_citations_in_corpus(EXAMPLE_PATH, window=5)

In [ ]:
citations_levenshtein = find_citations_in_corpus(EXAMPLE_PATH, window=5, method="levenstein", threshold=2)

In [ ]:
citations_levenshtein = find_citations_in_corpus(EXAMPLE_PATH, window=5, method="levenstein_preselect", threshold=2)